# AdaptiveSLM mobile baseline — Colab / Kaggle

Start with a pretrained model and a short LoRA run. This notebook is a reproducibility baseline, not evidence of agent ability or a new foundation model.

**Local validation:** numerical/data regressions, interrupted-run recovery, and HF merge/reload on CPU. **Phone pipeline check:** the synthetic smoke artifact ran on the M35 native CPU runtime. **Still required:** your actual Colab/Kaggle GPU, agent quality, full-app memory and sustained phone measurements.

Upload `adaptive-slm-mobile-starter.zip` alongside this notebook on Colab, or add it as a Kaggle input dataset. The zip carries these new files even before they are pushed to GitHub. Enable a GPU and internet on Kaggle. Run cells in order; errors retain their full tracebacks. No fixed training duration is promised.


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

# Optional: set the exact path to the uploaded starter zip, especially on Kaggle.
STARTER_ZIP = ""
ROOT = Path.cwd()
if not (ROOT / "finetune_mobile.py").exists() and (ROOT / "training/finetune_mobile.py").exists():
    ROOT = ROOT / "training"
if not (ROOT / "finetune_mobile.py").exists():
    candidates = [Path(STARTER_ZIP)] if STARTER_ZIP else list(ROOT.glob("adaptive-slm-mobile-starter.zip"))
    if not candidates and Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob("adaptive-slm-mobile-starter.zip"))
    if len(candidates) != 1:
        raise RuntimeError("Upload the starter zip and set STARTER_ZIP to its exact path. No GitHub pull is needed.")
    ROOT = Path("/kaggle/working/aslm-mobile") if Path("/kaggle/working").exists() else ROOT / "aslm-mobile"
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as archive:
        # The provided archive contains flat filenames plus tests/.
        for member in archive.infolist():
            destination = (ROOT / member.filename).resolve()
            if not destination.is_relative_to(ROOT.resolve()):
                raise RuntimeError("Archive contains a path outside the starter directory")
        archive.extractall(ROOT)
print("Starter:", ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-mobile.txt")], check=True)
# The 2026.07 Colab image includes an old optional TorchAO package that PEFT rejects.
# Select runtime image 2026.07 for this tested compatibility path.
import importlib.metadata
if "google.colab" in sys.modules and importlib.metadata.version("torch").startswith("2.11."):
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "torchao==0.17.0"], check=True)
print("Restart the notebook kernel after installation, then rerun this cell and continue if packages were already imported.")


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab/Kaggle GPU runtime before continuing")
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
print("native bf16:", torch.cuda.is_bf16_supported(including_emulation=False), "compute capability:", torch.cuda.get_device_capability(0))
# Confirm that the installed wheel can actually execute on this GPU.
x = torch.ones((16, 16), device="cuda", dtype=torch.float16)
assert torch.isfinite(x @ x).all().item()
subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "tests"), "-q"], check=True)


In [ ]:
# Keep initial source revisions fixed. The script also writes exact package,
# data, tokenizer-output and device metadata into the run manifest.
WORK = Path("/kaggle/working/aslm-runs") if Path("/kaggle/working").exists() else ROOT / "runs"
COMMON = [
    sys.executable, str(ROOT / "finetune_mobile.py"),
    "--model", "HuggingFaceTB/SmolLM2-360M-Instruct",
    "--model-revision", "a10cc1512eabd3dde888204e902eca88bddb4951",
    "--dataset", "HuggingFaceTB/smol-smoltalk",
    "--dataset-revision", "f73fe857d519ff6ac5af2ea67c4d3834da7b8bcc",
    "--max-samples", "2000", "--max-length", "512",
    "--batch-size", "1", "--grad-accum", "2", "--save-steps", "5",
    "--max-steps", "10", "--output-dir", str(WORK / "smoke"),
]
# Optional transfer from a previous session. Use only archives you created.
CHECKPOINT_ZIP = ""
if CHECKPOINT_ZIP:
    import hashlib, json
    RUN = WORK / "smoke"
    if list(RUN.glob("checkpoint-*")):
        raise RuntimeError("Restore into a run directory without existing checkpoints")
    with zipfile.ZipFile(CHECKPOINT_ZIP) as archive:
        hashes = json.loads(archive.read("checkpoint_sha256.json"))
        if set(archive.namelist()) != set(hashes) | {"checkpoint_sha256.json"}:
            raise RuntimeError("Checkpoint archive has unexpected or missing files")
        for name, expected in hashes.items():
            if not (RUN / name).resolve().is_relative_to(RUN.resolve()):
                raise RuntimeError("Checkpoint path escapes the run directory")
            sha = hashlib.sha256()
            with archive.open(name) as entry:
                while block := entry.read(1024 * 1024):
                    sha.update(block)
            if sha.hexdigest() != expected:
                raise RuntimeError("Checkpoint checksum failed: " + name)
        RUN.mkdir(parents=True, exist_ok=True)
        archive.extractall(RUN)
    print("Restored checkpoint into:", RUN)
subprocess.run(COMMON + ["--preflight-only"], check=True)


The preflight materializes the pinned dataset in a local Arrow cache before sampling. This takes disk space but avoids interrupted remote reader failures and can reuse the cached data. Overlength conversations are counted and skipped, so tool JSON is never partially trained. Read `preflight.json` before interpreting sample counts.

The starter uses generic conversation data that SmolLM2 has already encountered during its original post-training. Its validation loss is useful for pipeline checks; it cannot support a claim of new generalization or agent success. Use fresh held-out tasks for research.


In [ ]:
# Ten optimizer steps establish that this GPU can train and export.
# Rerunning resumes a compatible checkpoint. Changing the experiment requires
# a different output directory; the script rejects incompatible resumes.
subprocess.run(COMMON, check=True)


In [ ]:
# Run after a checkpoint has finished saving; persist/download this zip.
# Repeat at session boundaries. The merged model/adapter alone cannot resume.
TRANSFER = WORK / "checkpoint-transfer.zip"
subprocess.run([sys.executable, str(ROOT / "bundle_checkpoint.py"), str(WORK / "smoke"), str(TRANSFER)], check=True)
print("Save this archive outside the ephemeral runtime:", TRANSFER)


In [ ]:
# Inspect the merged HF artifact before converting it.
from transformers import AutoModelForCausalLM, AutoTokenizer
MERGED = WORK / "smoke/merged"
tok = AutoTokenizer.from_pretrained(MERGED)
model = AutoModelForCausalLM.from_pretrained(MERGED, dtype=torch.float16).to("cuda").eval()
inputs = tok.apply_chat_template(
    [{"role": "user", "content": "Explain photosynthesis in one sentence."}],
    tokenize=True, add_generation_prompt=True, enable_thinking=False,
    return_dict=True, return_tensors="pt",
).to("cuda")
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tok.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
del model
torch.cuda.empty_cache()


## Optional conversion for this SmolLM2 baseline

The conversion code below is pinned to llama.cpp `b5260` to match the existing runtime. This pin is for the Llama-family SmolLM2 starter. **LFM2.5 and Qwen3.5 need a newer, separately validated runtime and their correct chat templates.** Do not send their artifacts to the existing b5260 wrapper.


In [ ]:
LLAMA = WORK / "llama.cpp-b5260"
if not LLAMA.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "b5260",
                    "https://github.com/ggml-org/llama.cpp.git", str(LLAMA)], check=True)
print("Converter commit:", subprocess.check_output(["git", "-C", str(LLAMA), "rev-parse", "HEAD"], text=True).strip())
# The old converter has incompatible Transformers requirements. Keep its
# dependencies separate and reuse the cloud provider's CUDA PyTorch wheel.
CONVERTER_ENV = WORK / "converter-venv"
CONVERTER_PY = CONVERTER_ENV / "bin/python"
if not CONVERTER_PY.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(CONVERTER_ENV)], check=True)
subprocess.run([str(CONVERTER_PY), "-m", "pip", "install", "-r", str(ROOT / "requirements-converter.txt")], check=True)
subprocess.run([str(CONVERTER_PY), "-c", "import torch; print('Converter torch:', torch.__version__)"], check=True)
subprocess.run(["cmake", "-S", str(LLAMA), "-B", str(LLAMA / "build"),
                "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF"], check=True)
subprocess.run(["cmake", "--build", str(LLAMA / "build"), "--target", "llama-quantize", "--parallel", "2"], check=True)
CONVERTER_INPUT = WORK / "converter-input"
subprocess.run([sys.executable, str(ROOT / "prepare_legacy_converter.py"), str(MERGED), str(CONVERTER_INPUT)], check=True)
F16 = WORK / "mobile-smoke-f16.gguf"
Q4 = WORK / "mobile-smoke-q4_k_m.gguf"
subprocess.run([str(CONVERTER_PY), str(LLAMA / "convert_hf_to_gguf.py"), str(CONVERTER_INPUT),
                "--outtype", "f16", "--outfile", str(F16)], check=True)
subprocess.run([str(LLAMA / "build/bin/llama-quantize"), str(F16), str(Q4), "Q4_K_M"], check=True)
print("GGUF:", Q4, "MiB:", Q4.stat().st_size / 2**20)


## Keep outputs before the runtime disappears

Save the GGUF, merged HF directory, adapter, run_manifest.json, eval_results.json, and checkpoint directories. Checkpoints contain optimizer/scheduler/RNG state; the adapter alone cannot resume training exactly. Colab ephemeral storage disappears on runtime reset. On Kaggle, save an output version or download the artifacts. Save to persistent storage between runs.

Next, benchmark an unmodified pretrained baseline and this merged/quantized artifact on the same phone. A ten-step smoke run is not expected to improve model quality. Train on your custom app-tool examples only after the tool schema, held-out evaluation, language requirements, and memory target are defined.
